# GPU session — two jobs, run top to bottom

**Part A (~20 min): the 650M capacity-matched control.** Closes the last unresolved claim from audit M4.
The 8M-scale control has already been run and the sparse-decomposition advantage **did not replicate**
(SAE AP 0.688 vs dense 0.735 vs sparsity-matched dense 0.837; margin over a random ReLU projection
+0.066 against a ±0.133 seed spread). The manuscript currently reports the 650M line as untested.
This part settles it.

**Part B (~10 h): the holo MD rescue.** Parameterizes SLU so the ligand is actually present, then runs
3 systems × 3 replicates × 3.3 ns.

> **Order is not optional.** Part A uses Colab's stock PyTorch. Part B calls `condacolab.install()`,
> which **replaces the Python environment and restarts the runtime** — anything installed before it is
> gone. Part A writes its results to `results_650m_control.csv` on disk, which survives the restart.
> Run Part A first, confirm the table printed, then continue into Part B.

---

## Part A — 650M capacity-matched control

Same protocol as the 8M control, same evaluation set: the audit-corrected **strict** benchmark
(27 click-beetle positives = 4.99 effective clusters vs. **32 genuinely labelled** negatives,
cluster-weighted, AP baseline 0.1849; the 59 homology-only *Ignelater* paralogs are excluded).

Two published 650M claims are tested:

1. `AP 0.470 → 0.590` for dense → SAE
2. *Ignelater* genome rank `#7 → #1` for the true luciferase among 61 PF00501 paralogs

against four controls: dense, dense at matched sparsity, random Gaussian projection to 10,240-d, and
random ReLU projection to 10,240-d (overcomplete + sparse + non-negative, but **not learned**).

Upload `closeout_inputs.zip` (contains `dataset.tsv` and `scores.tsv`).

In [ ]:
import os, io, zipfile, numpy as np, pandas as pd, torch
import time, sys
_T0 = time.time()
def log(msg, **kw):
    """Timestamped, force-flushed. Colab buffers stdout, so flush is not optional."""
    el = time.time() - _T0
    print(f"[{int(el)//60:3d}m{int(el)%60:02d}s] {msg}", flush=True, **kw)


if not os.path.exists("dataset.tsv"):
    from google.colab import files
    up = files.upload()
    for k in up:
        if k.endswith(".zip"): zipfile.ZipFile(io.BytesIO(up[k])).extractall(".")
        else: open(k, "wb").write(up[k])
for root, _, fs in os.walk("."):
    for f in fs:
        if f in ("dataset.tsv", "scores.tsv"): os.replace(os.path.join(root, f), f)

data = pd.read_csv("dataset.tsv", sep="\t"); S = pd.read_csv("scores.tsv", sep="\t")
seqs = dict(zip(data.accession, data.sequence))
print(f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# ---- embed with ESM-2 650M, layer 18, + InterPLM-650M SAE -----------------------------
from transformers import AutoTokenizer, EsmModel
from huggingface_hub import hf_hub_download

GPU = torch.cuda.is_available()
if not GPU:
    print("WARNING: no GPU attached. Runtime > Change runtime type > T4 GPU.\n"
          "         650M on CPU will take roughly an hour instead of ~5 min.")
dev = "cuda" if GPU else "cpu"
tok = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
esm = EsmModel.from_pretrained("facebook/esm2_t33_650M_UR50D", add_pooling_layer=False).eval()
esm = (esm.half() if GPU else esm.float()).to(dev)   # fp16 only on CUDA
st = torch.load(hf_hub_download("Elana/InterPLM-esm2-650m", "layer_18/ae_normalized.pt"),
                map_location="cpu")
W_enc, b_enc, b_pre = (st["encoder.weight"].float().to(dev),
                       st["encoder.bias"].float().to(dev),
                       st["bias"].float().to(dev))
print("SAE:", tuple(W_enc.shape), "(features x d_model)")

accs = list(S.accession)
Xd = np.zeros((len(accs), 1280), np.float32)
Xs = np.zeros((len(accs), W_enc.shape[0]), np.float32)
log(f"embedding {len(accs)} sequences through ESM-2 650M (layer 18) + SAE ...")
_t = time.time()
with torch.no_grad():
    for i, a in enumerate(accs):
        t_ = tok(seqs[a][:1022], return_tensors="pt").to(dev)
        h = esm(**t_, output_hidden_states=True).hidden_states[18][0, 1:-1].float()  # strip cls/eos
        Xd[i] = h.mean(0).cpu().numpy()
        Xs[i] = torch.relu((h - b_pre) @ W_enc.T + b_enc).mean(0).cpu().numpy()
        if (i+1) % 25 == 0 or i+1 == len(accs):
            rate = (i+1)/(time.time()-_t)
            eta  = (len(accs)-i-1)/max(rate, 1e-9)
            log(f"  {i+1:4d}/{len(accs)}  ({100*(i+1)/len(accs):5.1f}%)  "
                f"{rate:5.1f} seq/s  ETA {int(eta)//60:d}m{int(eta)%60:02d}s")
del esm; torch.cuda.empty_cache()
log(f"done. dense {Xd.shape} | SAE {Xs.shape}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

idx = {a: i for i, a in enumerate(accs)}
tr_pos = S[(S.label=="positive") & (S.split_group=="firefly")]
tr_neg = S[(S.label=="negative") & (S.split_group=="nonlum_other") &
           (~S.genus.isin(["Agrilus","Tenebrio","Leptinotarsa","Nicrophorus"]))]
pos = S[(S.label=="positive") & (S.split_group=="click_beetle")]
neg = S[(S.split_group=="nonlum_elateroid") |
        ((S.split_group=="nonlum_other") & (S.genus=="Agrilus"))]
ign = S[S.genus == "Ignelater"]; TRUE_LUC = {"A0A8K0DA02", "A0A8K0D8G1"}

tr = np.array([idx[a] for a in list(tr_pos.accession)+list(tr_neg.accession)])
te = np.array([idx[a] for a in list(pos.accession)+list(neg.accession)])
ig = np.array([idx[a] for a in ign.accession])
y_tr = np.r_[np.ones(len(tr_pos)), np.zeros(len(tr_neg))]
y_te = np.r_[np.ones(len(pos)),    np.zeros(len(neg))]
w_tr = np.r_[tr_pos.weight.values, tr_neg.weight.values]
w_te = np.r_[pos.weight.values,    neg.weight.values]
BASE = pos.weight.sum()/(pos.weight.sum()+neg.weight.sum())
true_local = [k for k, a in enumerate(ign.accession) if a in TRUE_LUC]
print(f"TEST {len(pos)} pos ({pos.weight.sum():.2f} clusters) vs {len(neg)} neg "
      f"({neg.weight.sum():.2f}) | AP baseline {BASE:.4f} | Ignelater paralogs: {len(ign)}")

def evaluate(X, C=1.0):
    sc = StandardScaler().fit(X[tr])
    bw = w_tr.copy(); bw[y_tr==1] *= bw[y_tr==0].sum()/bw[y_tr==1].sum()
    lr = LogisticRegression(penalty="l1", solver="liblinear", C=C, max_iter=5000)
    lr.fit(sc.transform(X[tr]), y_tr, sample_weight=bw)
    s_te = lr.decision_function(sc.transform(X[te]))
    s_ig = lr.decision_function(sc.transform(X[ig]))
    order = np.argsort(-s_ig)
    ranks = sorted(int(np.where(order == k)[0][0]) + 1 for k in true_local)
    return dict(ap=average_precision_score(y_te, s_te, sample_weight=w_te),
                auroc=roc_auc_score(y_te, s_te, sample_weight=w_te),
                nnz=int((np.abs(lr.coef_[0]) > 1e-8).sum()), ranks=ranks)

D, K = Xd.shape[1], Xs.shape[1]
rows = []
log("[1/4] dense probe ...");            r = evaluate(Xd)
rows.append(("Dense ESM-2 650M (layer 18)", D, r, ""))
log("[2/4] SAE probe (10,240-d) ...");   r_sae = evaluate(Xs)
log(f"      SAE: AUROC {r_sae['auroc']:.3f}  AP {r_sae['ap']:.3f}  nnz {r_sae['nnz']}")
log("[3/4] dense at matched sparsity: 40-point C sweep ...")
best = None
for n_, C in enumerate(np.logspace(-3, 2, 40), 1):
    rr = evaluate(Xd, C)
    if best is None or abs(rr["nnz"]-min(r_sae["nnz"], D)) < abs(best["nnz"]-min(r_sae["nnz"], D)): best = rr
    if n_ % 10 == 0: log(f"      sweep {n_}/40  (best nnz so far {best['nnz']}, target {min(r_sae['nnz'], D)})")
rows.append(("Dense, matched sparsity", D, best, ""))
log("[4/4] random projections: 2 variants x 10 seeds on 10,240-d (the slow part) ...")
for name, relu in [("Random Gaussian projection", False), ("Random ReLU projection", True)]:
    accum = []
    for seed in range(10):
        P = np.random.default_rng(seed).normal(0, 1/np.sqrt(D), (D, K)).astype(np.float32)
        Z = Xd @ P
        accum.append(evaluate(np.maximum(Z, 0) if relu else Z))
        log(f"      {name[:18]:18s} seed {seed+1}/10  AP {accum[-1]['ap']:.3f}")
    agg = dict(ap=np.mean([a["ap"] for a in accum]), auroc=np.mean([a["auroc"] for a in accum]),
               nnz=0, ranks=sorted(np.median([a["ranks"] for a in accum], axis=0).astype(int)))
    rows.append((f"{name} (10 seeds)", K, agg, f"+/- {np.std([a['ap'] for a in accum], ddof=1):.3f}"))
rows.append(("InterPLM-650M SAE (layer 18)", K, r_sae, ""))

print(f"\nAP baseline (cluster-weighted chance) = {BASE:.4f}\n")
print(f"{'representation':36s} {'dim':>6s} {'AUROC':>7s} {'AP':>7s} {'nnz':>5s}  {'Ignelater ranks':>16s}")
print("-"*86)
for nm, dim, rr, sd in rows:
    print(f"{nm:36s} {dim:6d} {rr['auroc']:7.3f} {rr['ap']:7.3f} "
          f"{(rr['nnz'] or '-'):>5} {str(rr['ranks']):>16s}  {sd}")
print("-"*86)
gap = r_sae["ap"] - rows[3][2]["ap"]
sd_relu = float(rows[3][3].split()[-1])
print(f"\nSAE minus random ReLU projection = {gap:+.3f} AP   (control seed spread +/- {sd_relu:.3f})")
print("  Published 650M claims under test: AP 0.470 -> 0.590, Ignelater rank #7 -> #1")
print("  ->", "SAE advantage EXCEEDS the structure-free control" if gap > sd_relu else
      "SAE advantage is INSIDE the control's seed noise - claim not supported at 650M either")
pd.DataFrame([{"representation": nm, "dim": dim, "auroc": rr["auroc"], "ap": rr["ap"],
               "nnz": rr["nnz"], "ignelater_ranks": rr["ranks"], "ap_sd": sd}
              for nm, dim, rr, sd in rows]).to_csv("results_650m_control.csv", index=False)
print("\nSaved results_650m_control.csv (survives the condacolab restart). Paste it into §2.5 Table 3.")

---

## Part B — holo MD rescue

Everything from here needs the conda stack. `condacolab.install()` restarts the runtime once; that is
expected, and `results_650m_control.csv` from Part A persists on disk.

### Background: why the existing trajectories cannot be used

**Why this notebook exists.** The three 10 ns trajectories in `md_trajectories_10ns/` are **apo**. The previous
notebook dropped the ligand at read time:

```python
raw_lines = [l for l in fh if l.startswith(("ATOM", "TER", "END"))]   # <-- HETATM/SLU discarded
```

and earlier GROMACS/OpenMM attempts failed with
`Group Protein_SLU referenced in the .mdp file was not found in the index file` and
`No template found for residue 542 (SLU)`. Both failures have one cause: **SLU has no force-field parameters.**

This notebook generates them on the fly with OpenFF/SMIRNOFF so the ligand is present in the simulated
system. Only then can the L347/S347 steric-gate hypothesis actually be tested.

**Ligand identity** (RCSB chemical component `SLU`): 5'-O-[N-(dehydroluciferyl)-sulfamoyl] adenosine (DLSA),
C21 H18 N8 O8 S3, 40 heavy atoms, InChIKey `LJLYTUYNVSHXQB-SOONXTGKSA-N`. DLSA is the non-hydrolysable
analogue of luciferyl-adenylate, the intermediate bound in PDB `4G36`.

**Runtime.** Measured throughput on a Colab T4 was 74.5 ns/day, so 3 x 10 ns is about 10 GPU-hours. Run overnight.

## 1. Environment

`condacolab.install()` restarts the kernel once - that is expected. Resume at cell 2.

In [ ]:
!pip install -q condacolab
import condacolab; condacolab.install()   # kernel restarts here

In [ ]:
import condacolab; condacolab.check()
# Pin the OpenMM CUDA build to the DRIVER's CUDA version. conda-forge's default openmm is
# compiled against a newer toolkit than Colab's driver supports, which fails at runtime with
# CUDA_ERROR_UNSUPPORTED_PTX_VERSION (222) - after setup has already burned several minutes.
import subprocess, re
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
m = re.search(r"CUDA Version:\s*([0-9]+\.[0-9]+)", smi)
CUDA_V = m.group(1) if m else "12.4"
print(f"driver CUDA version: {CUDA_V}")
!mamba install -y -q -c conda-forge openmm cuda-version={CUDA_V} openmmforcefields openff-toolkit ambertools rdkit mdtraj
# openff-toolkit lazily probes the NAGL charge backend on first import. NAGL pulls in
# pytorch-lightning -> torchmetrics -> torchvision, and Colab's preinstalled torchvision
# clashes with the conda torch, raising "RuntimeError: operator torchvision::nms does not
# exist". openff only catches ImportError there, so the RuntimeError escapes and blocks
# `from openff.toolkit import Molecule` entirely. We do not use NAGL (charges come from
# AmberTools sqm via AM1-BCC), so remove it: the probe then fails with a clean ImportError
# that openff handles.
!pip uninstall -y openff-nagl pytorch-lightning torchmetrics 2>/dev/null | tail -2
try:
    from openff.toolkit import Molecule
    print("openff.toolkit imports cleanly")
except Exception as e:
    print("openff import still failing:", type(e).__name__, str(e)[:120])

import openmm
print("OpenMM", openmm.__version__, "| platforms:",
      [openmm.Platform.getPlatform(i).getName() for i in range(openmm.Platform.getNumPlatforms())])

## 2. Inputs

Upload the three holo PDBs from `results/md_inputs/` - they already contain protein `ATOM` **and** `SLU` `HETATM` records.

> **Check complex 3 first.** Its `ASN A 345` side chain was written as three hard-coded Cartesian coordinates by
> `scripts/fix_complex3.py`, not placed by a modelling tool. Cell 4 reports its local geometry; rebuild the
> rotamer if anything is out of range.

In [ ]:
import os, glob, re, shutil, zipfile, io

# ---------------------------------------------------------------------------------------
# Inputs. This cell is deliberately paranoid: a stale or browser-duplicated PDB in inputs/
# previously caused a 6-minute AM1-BCC run to be thrown away on a malformed file.
#   FORCE_REUPLOAD = True  -> wipe inputs/ and ask for the zip again (use after any fix)
#   FORCE_REUPLOAD = False -> reuse what is already in inputs/, but still validate it
# ---------------------------------------------------------------------------------------
FORCE_REUPLOAD = True

if FORCE_REUPLOAD and os.path.isdir("inputs"):
    shutil.rmtree("inputs")
    print("cleared inputs/ - upload the current holo_md_inputs.zip when prompted")
os.makedirs("inputs", exist_ok=True)

if not glob.glob("inputs/*.pdb"):
    from google.colab import files
    up = files.upload()
    for k, v in up.items():
        if k.lower().endswith(".zip"):
            zipfile.ZipFile(io.BytesIO(v)).extractall("inputs")
        else:
            open(os.path.join("inputs", k), "wb").write(v)

# Strip browser duplicate suffixes:  "name (1).pdb" -> "name.pdb"  (the later copy wins)
for p in sorted(glob.glob("inputs/*.pdb")):
    m = re.match(r"^(.*?) \(\d+\)(\.pdb)$", os.path.basename(p))
    if m:
        tgt = os.path.join("inputs", m.group(1) + m.group(2))
        print(f"  de-duplicating {os.path.basename(p)} -> {os.path.basename(tgt)}")
        os.replace(p, tgt)

COMPLEXES = sorted(glob.glob("inputs/*.pdb"))

# ---- validate EVERY file against the PDB column spec before anything expensive runs ----
problems = []
for p in COMPLEXES:
    n_slu = 0
    for ln, l in enumerate(open(p), 1):
        if l.startswith(("ATOM", "HETATM")):
            if len(l.rstrip("\n")) < 78:
                problems.append((p, ln, "line shorter than 78 cols", l)); break
            if len(l[17:20].strip()) != 3:
                problems.append((p, ln, f"residue name not 3 chars at cols 18-20: {l[17:20]!r}", l)); break
            if l[16] not in " AB":
                problems.append((p, ln, f"altLoc col 17 is {l[16]!r}, expected ' ' - residue name is shifted", l)); break
            try:
                float(l[30:38]); float(l[38:46]); float(l[46:54])
            except ValueError:
                problems.append((p, ln, "unparseable coordinates at cols 31-54", l)); break
            if l.startswith("HETATM") and l[17:20].strip() == "SLU":
                n_slu += 1
    if n_slu != 40 and not any(pr[0] == p for pr in problems):
        problems.append((p, 0, f"expected 40 SLU HETATM atoms, found {n_slu}", ""))

print(f"\n{len(COMPLEXES)} input structures:")
for i, c in enumerate(COMPLEXES):
    bad = [pr for pr in problems if pr[0] == c]
    n_at = sum(l.startswith("ATOM") for l in open(c))
    n_slu = sum(l.startswith("HETATM") and l[17:20].strip() == "SLU" for l in open(c))
    print(f"   [{i}] {os.path.basename(c):55s} ATOM={n_at:5d} SLU={n_slu:3d} "
          f"{'OK' if not bad else 'MALFORMED'}")
if problems:
    print("\n" + "!" * 86)
    for p, ln, why, l in problems:
        print(f"  {os.path.basename(p)}  line {ln}: {why}")
        if l: print(f"    {l.rstrip()}")
    print("!" * 86)
    raise SystemExit("Malformed input(s). Download the current holo_md_inputs.zip, set "
                     "FORCE_REUPLOAD=True, re-run this cell. Nothing expensive has run yet.")
print("\nall inputs valid - safe to proceed")

## 3. Geometry check on the hand-edited Asn (complex 3)

In [ ]:
import numpy as np
def atoms_of(pdb, resnum, resname):
    d = {}
    for l in open(pdb):
        if l.startswith("ATOM") and l[17:20].strip()==resname and int(l[22:26])==resnum:
            d[l[12:16].strip()] = np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])])
    return d
p3 = [c for c in COMPLEXES if "SAESteered" in c][0]
a = atoms_of(p3, 345, "ASN")
if a:
    dist = lambda x,y: float(np.linalg.norm(a[x]-a[y]))
    def ang(x,y,z):
        v1, v2 = a[x]-a[y], a[z]-a[y]
        return float(np.degrees(np.arccos(v1@v2/(np.linalg.norm(v1)*np.linalg.norm(v2)))))
    checks = [("CB-CG",dist("CB","CG"),1.52,0.05), ("CG-OD1",dist("CG","OD1"),1.23,0.04),
              ("CG-ND2",dist("CG","ND2"),1.33,0.04), ("CA-CB-CG",ang("CA","CB","CG"),113.4,6.0),
              ("OD1-CG-ND2",ang("OD1","CG","ND2"),122.0,6.0), ("CB-CG-OD1",ang("CB","CG","OD1"),120.8,6.0)]
    ok = True
    for nm, val, ideal, tol in checks:
        good = abs(val-ideal) <= tol; ok &= good
        print(f"  {nm:12s} {val:7.2f}  (ideal {ideal:6.2f} +/- {tol})  {'OK' if good else '<-- OUT OF RANGE'}")
    print("\nASN 345 geometry:", "acceptable" if ok else "REBUILD THIS ROTAMER BEFORE PRODUCTION")
    assert ok, ("ASN 345 side chain fails stereochemistry. Use the corrected "
                "holo_md_inputs.zip (chi1=-65, chi2=30, rebuilt from ideal internal "
                "coordinates); the hand-placed original is kept as *.handplaced_backup.")
else:
    print("ASN 345 not found - check residue numbering in complex 3")

## 4. Parameterize SLU

PDB files carry no bond orders, so they are assigned from the reference SMILES with
`AssignBondOrdersFromTemplate`; hydrogens are then added at the experimental heavy-atom coordinates and the
molecule handed to OpenFF for SMIRNOFF typing with AM1-BCC charges.

**Protonation choice.** The acyl-sulfamate N-H (`C(=O)-NH-S(=O)2-O-`) is strongly acidic (pKa ~2-4), so at pH 7
this nitrogen is deprotonated and the ligand carries **-1**. That also reproduces the -1 formal charge of the
luciferyl-adenylate phosphoanhydride it mimics, so the anion is the default. The neutral (as-deposited) form is
one switch away - if the gate result is load-bearing, run both and report the comparison.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from openff.toolkit import Molecule

PROTONATION = "anion"      # "anion" (recommended, -1) or "neutral" (0, as deposited)
SMILES = {
 "neutral": "c1cc2c(cc1O)sc(n2)c3nc(cs3)C(=O)NS(=O)(=O)OC[C@@H]4[C@H]([C@H]([C@@H](O4)n5cnc6c5ncnc6N)O)O",
 "anion":   "c1cc2c(cc1O)sc(n2)c3nc(cs3)C(=O)[N-]S(=O)(=O)OC[C@@H]4[C@H]([C@H]([C@@H](O4)n5cnc6c5ncnc6N)O)O",
}[PROTONATION]

def build_slu(pdb, tmp="slu.pdb"):
    with open(tmp, "w") as fh:
        for l in open(pdb):
            if l.startswith("HETATM") and l[17:20].strip()=="SLU": fh.write(l)
        fh.write("END\n")
    rd = Chem.MolFromPDBFile(tmp, removeHs=False, sanitize=False)
    rd = AllChem.AssignBondOrdersFromTemplate(Chem.MolFromSmiles(SMILES), rd)
    rd = Chem.AddHs(rd, addCoords=True)
    mol = Molecule.from_rdkit(rd, allow_undefined_stereo=False)
    mol.name = "SLU"
    return mol

slu = build_slu(COMPLEXES[0])
print("SLU atoms (with H):", slu.n_atoms, "| formal charge:", slu.total_charge)
slu.assign_partial_charges(partial_charge_method="am1bcc")     # ~6 min: sqm is single-threaded and Colab gives ~2 vCPUs
print(f"AM1-BCC charges assigned: sum = {slu.partial_charges.magnitude.sum():+.4f} e")

## 5. Build, solvate, equilibrate, run 3 replicates x 3.3 ns per complex

In [ ]:
from openmmforcefields.generators import SMIRNOFFTemplateGenerator
from mdtraj.reporters import DCDReporter as MDTrajDCDReporter
import openmm as mm, openmm.app as app, openmm.unit as unit
import mdtraj as mdt
from sys import stdout
import time, os, struct, json
import numpy as np

_T0 = time.time()
def log(msg):
    el = time.time() - _T0
    print(f"[{int(el)//3600:d}h{(int(el)//60)%60:02d}m] {msg}", flush=True)

PROTEIN = {"ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE","LEU","LYS",
           "MET","PHE","PRO","SER","THR","TRP","TYR","VAL",
           "HID","HIE","HIP","CYX","ASH","GLH","LYN","ACE","NME","NH2"}
SOLVENT = {"HOH","WAT","SOL","TIP","NA","CL","K","MG","ZN","CA","SOD","CLA"}

# ---- budget -------------------------------------------------------------------------
NS, DT_FS, REPORT = 2.5, 4.0, 2500      # 10 ps frames -> 250 frames per replicate
N_REPLICATES = 3
n_steps   = int(NS*1e6/DT_FS)
TOTAL_RUNS = len(COMPLEXES) * N_REPLICATES
SPEED_GUESS = 74.5                       # ns/day measured on a T4 for the apo systems
est_h = TOTAL_RUNS*NS/SPEED_GUESS*24 + (7 + (TOTAL_RUNS-1)*1.5)/60
run_times = []

def pick_platform():
    """Smoke-test each platform on a 1-particle system. Registration is NOT enough: a CUDA
    platform can be listed and still fail with CUDA_ERROR_UNSUPPORTED_PTX_VERSION when the
    conda build's toolkit is newer than the driver. Catch that here, in seconds, rather than
    after ~7 min of ligand prep and solvation."""
    for name, props in (("CUDA", {"Precision": "mixed"}),
                        ("OpenCL", {"Precision": "mixed"}),
                        ("CPU", {})):
        try:
            p = mm.Platform.getPlatformByName(name)
            s_ = mm.System(); s_.addParticle(1.0)
            ctx = mm.Context(s_, mm.VerletIntegrator(0.001), p, props) if props \
                  else mm.Context(s_, mm.VerletIntegrator(0.001), p)
            ctx.setPositions([[0, 0, 0]] * 1)
            ctx.getState(getEnergy=True)
            del ctx
            return p, props, name
        except Exception as e:
            print(f"  {name:7s} unusable: {str(e).splitlines()[0][:90]}", flush=True)
    raise RuntimeError("No usable OpenMM platform.")

PLATFORM, PLATFORM_PROPS, PLATFORM_NAME = pick_platform()
if PLATFORM_NAME != "CUDA":
    print(f"\n  !! Running on {PLATFORM_NAME}, not CUDA. CPU is ~50x slower - do not start a\n"
          f"     9-run session on it. Fix the driver/toolkit mismatch first (re-run the install\n"
          f"     cell, which pins cuda-version to the driver), or switch runtime to T4 GPU.\n")
log(f"platform: {PLATFORM_NAME}")
log(f"PLAN: {len(COMPLEXES)} systems x {N_REPLICATES} replicates x {NS} ns = {TOTAL_RUNS*NS:.1f} ns")
log(f"      estimated wall time ~{est_h:.1f} h at {SPEED_GUESS} ns/day")
log(f"      RESUME IS ON - completed runs are skipped, so a disconnect is not fatal.")
log(f"      Trajectories store protein+ligand only (water stripped): ~35 MB each, not ~400 MB.")

def dcd_frames(path):
    """Frame count from a DCD header, without loading coordinates."""
    try:
        with open(path, "rb") as f:
            f.seek(8); return struct.unpack("<i", f.read(4))[0]
    except Exception:
        return -1

# AM1-BCC charge cache, PERSISTED TO DISK.
# sqm is single-threaded and Colab gives ~2 vCPUs, so this takes ~6 min per call (not the
# 6 min quoted in older versions of this notebook). It is the same molecule in all three
# complexes, so it is computed once and reused. Writing it to disk means the cost also
# survives a kernel restart and is not re-paid when each complex is run in its own session.
CHARGE_FILE = "slu_am1bcc_charges.json"

def _ligand_key(lig, pdb_path):
    """Key on isomeric SMILES AND the ligand atom-name order read from the PDB. Charges are
    per-atom and positional, so reusing them across a different atom ordering would silently
    assign the wrong charge to every atom."""
    names = [l[12:16].strip() for l in open(pdb_path)
             if l.startswith("HETATM") and l[17:20].strip() == "SLU"]
    return lig.to_smiles() + "|" + ",".join(names)

def ligand_for(pdb_path):
    from openff.units import unit as offunit
    lig = build_slu(pdb_path)                      # conformer differs per complex
    key = _ligand_key(lig, pdb_path)
    disk = {}
    if os.path.exists(CHARGE_FILE):
        try: disk = json.load(open(CHARGE_FILE))
        except Exception: disk = {}
    if key in disk and len(disk[key]) == lig.n_atoms:
        lig.partial_charges = np.array(disk[key], dtype=float) * offunit.elementary_charge
        log(f"        AM1-BCC charges loaded from {CHARGE_FILE} (saved ~6 min)")
    else:
        log("        assigning AM1-BCC charges (sqm, single-threaded, ~6 min on Colab, silent) ...")
        lig.assign_partial_charges(partial_charge_method="am1bcc")
        disk[key] = [float(x) for x in lig.partial_charges.m]
        json.dump(disk, open(CHARGE_FILE, "w"))
        log(f"        charges cached to {CHARGE_FILE} - later complexes and sessions skip this")
    q = float(np.sum(lig.partial_charges.m))
    assert abs(q - lig.total_charge.m) < 1e-3, f"charge sum {q:.4f} != formal {lig.total_charge.m}"
    return lig

def run(pdb_path, replicate, run_no):
    base = os.path.basename(pdb_path).replace("_SLU_holo.pdb","").replace(".pdb","")
    tag = f"{base}_holo_rep{replicate}"
    os.makedirs(tag, exist_ok=True)
    want = n_steps // REPORT

    log("="*74)
    log(f"RUN {run_no}/{TOTAL_RUNS}  |  {base[:40]}  |  replicate {replicate+1}/{N_REPLICATES}")
    have = dcd_frames(f"{tag}/prod.dcd")
    if have >= want and os.path.exists(f"{tag}/top.pdb"):
        log(f"   ALREADY COMPLETE ({have}/{want} frames) - skipping."); log("="*74); return
    if have > 0:
        log(f"   partial trajectory found ({have}/{want} frames) - restarting this run from scratch")
    if run_times:
        rem = (TOTAL_RUNS - run_no + 1) * (sum(run_times)/len(run_times))
        log(f"   projected remaining for the session: {rem/3600:.1f} h")
    log("="*74)
    t_run = time.time()

    log("  [1/7] building SLU ...")
    lig = ligand_for(pdb_path)
    log(f"        SLU: {lig.n_atoms} atoms, formal charge {lig.total_charge}")

    log("  [2/7] registering SMIRNOFF template generator ...")
    gen = SMIRNOFFTemplateGenerator(molecules=[lig], forcefield="openff-2.2.0")
    ff = app.ForceField("amber14-all.xml", "amber14/tip3p.xml")
    ff.registerTemplateGenerator(gen.generator)

    log("  [3/7] protonating protein, then appending ligand as ONE residue ...")
    with open("prot.pdb","w") as fh:
        for l in open(pdb_path):
            if l.startswith(("ATOM","TER")): fh.write(l)
        fh.write("END\n")
    prot = app.PDBFile("prot.pdb")
    mod = app.Modeller(prot.topology, prot.positions)
    # Protonate the PROTEIN ONLY, before the ligand is present. The ligand already carries
    # explicit hydrogens from Chem.AddHs, so addHydrogens must not see it.
    mod.addHydrogens(ff, pH=7.0)

    # Rebuild the ligand topology as a SINGLE residue. openff's to_openmm() inherits PDB
    # residue metadata per atom: the 40 heavy atoms keep "SLU" from the input file while the
    # 17 hydrogens added by Chem.AddHs have none and land in a separate "UNK" residue. That
    # splits one molecule across two residue records, which trips the ligand guard and breaks
    # residue-based selection downstream. Chemistry is unaffected (SMIRNOFF types by
    # connectivity, not residue), so we fix the bookkeeping rather than relax the check.
    _lt = lig.to_topology().to_openmm()
    lig_top = app.Topology()
    _res = lig_top.addResidue("SLU", lig_top.addChain())
    _map = {}
    for a in _lt.atoms():
        _map[a.index] = lig_top.addAtom(a.name, a.element, _res)
    for b in _lt.bonds():
        lig_top.addBond(_map[b[0].index], _map[b[1].index])
    assert lig_top.getNumResidues() == 1 and lig_top.getNumAtoms() == lig.n_atoms
    mod.add(lig_top, lig.conformers[0].to_openmm())

    log("  [4/7] solvating (TIP3P, 1.0 nm padding, 0.15 M NaCl, ~1 min, silent) ...")
    mod.addSolvent(ff, model="tip3p", padding=1.0*unit.nanometer, ionicStrength=0.15*unit.molar)

    # ---- LIGAND GUARD (name-agnostic; see notes) --------------------------------------
    lig_res = [r for r in mod.topology.residues()
               if r.name.strip().upper() not in PROTEIN | SOLVENT]
    n_lig_at = sum(1 for r in lig_res for _ in r.atoms())
    log(f"        system: {mod.topology.getNumAtoms()} atoms")
    log(f"        non-protein / non-solvent residues: {[r.name for r in lig_res]} "
        f"({n_lig_at} atoms, expected {lig.n_atoms})")
    assert len(lig_res) == 1, (f"LIGAND GUARD FAILED: expected 1 ligand residue, found "
        f"{len(lig_res)}: {[r.name for r in lig_res]}. 0 = ligand dropped (the original bug); "
        f">1 = split across residue records. Do not continue.")
    assert n_lig_at == lig.n_atoms, (f"LIGAND GUARD FAILED: ligand has {n_lig_at} atoms, "
        f"expected {lig.n_atoms} - present but mangled. Do not continue.")
    log(f"        LIGAND GUARD PASSED")

    # keep protein + ligand, drop water/ions from the stored trajectory
    keep = [a.index for a in mod.topology.atoms()
            if a.residue.name.strip().upper() not in SOLVENT]
    log(f"        storing {len(keep)} of {mod.topology.getNumAtoms()} atoms per frame "
        f"({100*len(keep)/mod.topology.getNumAtoms():.1f}%)")

    log("  [5/7] creating system (PME, HBonds, HMR 4 fs) ...")
    system = ff.createSystem(mod.topology, nonbondedMethod=app.PME,
                             nonbondedCutoff=1.0*unit.nanometer, constraints=app.HBonds,
                             rigidWater=True, hydrogenMass=4.0*unit.amu)
    system.addForce(mm.MonteCarloBarostat(1.0*unit.bar, 300*unit.kelvin))
    integ = mm.LangevinMiddleIntegrator(300*unit.kelvin, 1.0/unit.picosecond, DT_FS*unit.femtoseconds)
    sim = app.Simulation(mod.topology, system, integ, PLATFORM, PLATFORM_PROPS)
    sim.context.setPositions(mod.positions)

    log("  [6/7] minimizing (silent, up to a few min) ...")
    sim.minimizeEnergy()
    e = sim.context.getState(getEnergy=True).getPotentialEnergy()
    log(f"        E_pot = {e.value_in_unit(unit.kilojoule_per_mole):,.0f} kJ/mol")
    with open(f"{tag}/solvated_full.pdb","w") as fh:
        app.PDBFile.writeFile(sim.topology, sim.context.getState(getPositions=True).getPositions(), fh)
    mdt.load(f"{tag}/solvated_full.pdb").atom_slice(keep).save_pdb(f"{tag}/top.pdb")
    os.remove(f"{tag}/solvated_full.pdb")
    log(f"        wrote {tag}/top.pdb (matches the stored trajectory)")

    log(f"        equilibrating 200 ps (velocity seed {1234+replicate}) ...")
    sim.context.setVelocitiesToTemperature(300*unit.kelvin, 1234 + replicate)
    sim.step(50000)

    log(f"  [7/7] production {NS} ns -> {want} frames, status every 100 ps")
    sim.reporters.append(MDTrajDCDReporter(f"{tag}/prod.dcd", REPORT, atomSubset=keep))
    sim.reporters.append(app.StateDataReporter(stdout, 25000, step=True, time=True,
                         potentialEnergy=True, temperature=True, speed=True, progress=True,
                         remainingTime=True, elapsedTime=True, totalSteps=n_steps, separator="  "))
    sim.step(n_steps)
    for r in sim.reporters:
        if hasattr(r, "close"): r.close()
    dt_run = time.time() - t_run
    run_times.append(dt_run)
    log(f"  RUN {run_no}/{TOTAL_RUNS} done in {dt_run/60:.1f} min -> {tag}/prod.dcd "
        f"({dcd_frames(f'{tag}/prod.dcd')} frames, {os.path.getsize(f'{tag}/prod.dcd')/1e6:.0f} MB)")

def run_complex(ci):
    """Run all replicates of ONE complex. Independent of the other complexes: if this cell
    fails or the session drops, the other two are untouched, and re-running it skips any
    replicate whose trajectory is already complete."""
    pdb = COMPLEXES[ci]
    name = os.path.basename(pdb)
    log("#"*74)
    log(f"COMPLEX {ci+1}/{len(COMPLEXES)}: {name}")
    log("#"*74)
    t0 = time.time()
    for rep in range(N_REPLICATES):
        run(pdb, rep, ci*N_REPLICATES + rep + 1)
    done = [r for r in range(N_REPLICATES)
            if dcd_frames(f"{os.path.basename(pdb).replace('_SLU_holo.pdb','').replace('.pdb','')}"
                          f"_holo_rep{r}/prod.dcd") >= n_steps // REPORT]
    log(f"COMPLEX {ci+1} DONE in {(time.time()-t0)/3600:.2f} h "
        f"- {len(done)}/{N_REPLICATES} replicates complete")

log(f"ready. {len(COMPLEXES)} complexes detected:")
for j, cx in enumerate(COMPLEXES):
    log(f"   [{j}] {os.path.basename(cx)}")
log("Run each complex in its own cell below.")

### Run the complexes — one cell each

Each cell is **independent**. If complex 3 fails, complexes 1 and 2 are untouched and you do
not re-run them. Re-running any cell skips replicates that already finished, so a disconnect
mid-complex costs only the replicate that was in flight.

Expected: ~2.7 h per complex on a T4 (3 replicates x 2.5 ns).

In [ ]:
# COMPLEX 1: True luciferase (S347) — the functional reference
run_complex(0)

In [ ]:
# COMPLEX 2: Non-luminous paralog (L347) — the decoy
run_complex(1)

In [ ]:
# COMPLEX 3: SAE-steered mutant (S347/A348/N349) — causal test
run_complex(2)

### Complex 4 — POSITIVE CONTROL (S347F)

**Why this run exists.** Complexes 1–3 showed no steric occlusion. A negative result is only
meaningful if the assay can detect occlusion *when it is really there*. This control places a
bulky phenylalanine at the gate of the true luciferase, in an ideal rotamer
($\chi_1 = -177^\circ$, $\chi_2 = 80^\circ$) with no backbone clash (2.33 Å to the nearest
protein atom) but with the phenyl ring **1.62 Å from the ligand** — versus 3.56 Å for the
wild-type serine hydroxyl.

**How to read the outcome:**

| Result | Meaning |
|---|---|
| S347F occludes (substantial frames < 2.8 Å) | the assay is sensitive → the L347 negative result is real and load-bearing |
| S347F also shows no occlusion | the assay cannot detect occlusion at all → §2.3b proves nothing and must be withdrawn |

Run this before drawing any conclusion from complexes 1–3.

In [ ]:
# COMPLEX 4: POSITIVE CONTROL — S347F bulky-gate mutant of the true luciferase
run_complex(3)

## 6. The actual experiment: gate-to-ligand distance

This is the measurement the project has been missing. Residue numbering differs between topologies, so the
gate is found by **motif anchoring** against *P. pyralis* P08659 rather than a hard-coded index
(`A0A8K0DA02` has it at 346-348, `A0A8K0C7P5` at 343-345). The hypothesis is that bulky L347 holds the
minimum side-chain-to-ligand distance below the ~2.8 A clash threshold while S347 does not.

In [ ]:
import mdtraj as mdt, numpy as np, matplotlib.pyplot as plt, collections, re, os

AA3 = {"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G","HIS":"H","ILE":"I",
       "LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V",
       "HID":"H","HIE":"H","HIP":"H","CYX":"C"}
SOLVENT = {"HOH","WAT","SOL","TIP","NA","CL","K","MG","ZN","CA","SOD","CLA"}

def gate_residues(top):
    """P08659-aligned 347/348/349 via the GLTE anchor (verified unique in all 3 topologies)."""
    res = [r for r in top.residues if r.name.strip().upper() in AA3]
    seq = "".join(AA3[r.name.strip().upper()] for r in res)
    hits = [i for i in range(len(seq)-3) if seq[i:i+4] == "GLTE"]
    assert len(hits) == 1, f"GLTE anchor not unique: {hits}"
    s = hits[0] + 6
    return [res[s+i].index for i in range(3)], seq[s:s+3]

def ligand_atoms(top):
    """Heavy atoms of the ligand. NAME-AGNOSTIC: openff/OpenMM may call the residue UNK,
    not SLU, so select by exclusion (not protein, not solvent) exactly as the run-time
    guard does. Testing `a.residue.name == "SLU"` here would silently return [] and
    destroy the analysis after the simulations had already finished."""
    lig = [r for r in top.residues
           if r.name.strip().upper() not in set(AA3) | SOLVENT]
    assert lig, "no ligand residue found - the trajectory is apo"
    # Take the UNION across ligand residue records. The MD cell writes the ligand as a single
    # residue, but accept a split (e.g. SLU + UNK from Chem.AddHs hydrogens) because the
    # chemically meaningful check is the heavy-atom count, not the residue bookkeeping.
    idx = [a.index for r in lig for a in r.atoms if a.element.symbol != "H"]
    assert len(idx) == 40, (f"ligand should have 40 heavy atoms, found {len(idx)} "
                            f"across residues {[r.name for r in lig]}")
    return idx, "+".join(sorted({r.name for r in lig}))

def min_gate_dist(tag):
    tr = mdt.load(f"{tag}/prod.dcd", top=f"{tag}/top.pdb")
    gidx, gaa = gate_residues(tr.topology)
    lig_at, lig_name = ligand_atoms(tr.topology)
    bb = {"N","CA","C","O"}
    gate_at = [a.index for r in gidx for a in tr.topology.residue(r).atoms
               if a.name not in bb and a.element.symbol != "H"]
    assert gate_at, "no gate side-chain heavy atoms selected"
    pairs = np.array([(i, j) for i in gate_at for j in lig_at])
    d = mdt.compute_distances(tr, pairs).min(axis=1) * 10.0     # nm -> Angstrom
    return d, gaa, lig_name

runs = sorted(d for d in os.listdir(".") if re.search(r"_holo_rep\d+$", d)
              and os.path.isdir(d) and os.path.exists(f"{d}/prod.dcd"))
assert runs, "no completed runs found"
by_system = collections.defaultdict(list)
for tag in runs:
    v, gaa, lig_name = min_gate_dist(tag)
    by_system[re.sub(r"_rep\d+$", "", tag)].append(v)
    print(f"{tag[:50]:50s} gate={gaa} lig={lig_name:4s} n={len(v):4d}  "
          f"min={v.min():5.2f}  median={np.median(v):5.2f}  frac(<2.8 A)={np.mean(v<2.8):.3f}", flush=True)

print("\n" + "="*84)
print(f"{'system':44s} {'median dist (A)':>22s}  {'frac < 2.8 A':>16s}")
print("="*84)
for sysname, reps in by_system.items():
    med  = np.array([np.median(r) for r in reps])
    frac = np.array([np.mean(r < 2.8) for r in reps])
    sd_m = med.std(ddof=1) if len(med) > 1 else float("nan")
    sd_f = frac.std(ddof=1) if len(frac) > 1 else float("nan")
    print(f"{sysname[:44]:44s} {med.mean():8.2f} +/- {sd_m:5.2f} (n={len(reps)}) "
          f"{frac.mean():8.3f} +/- {sd_f:.3f}")
print("="*84)
print("The +/- is the spread ACROSS INDEPENDENT REPLICATES - the error bar the single-run")
print("apo trajectories could not provide. Report it, and do not over-read a difference")
print("between systems that is smaller than it.")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for sysname, reps in by_system.items():
    ax[0].hist(np.concatenate(reps), bins=50, density=True, alpha=.5, label=sysname[:30])
    ax[1].scatter([sysname[:16]]*len(reps), [np.median(r) for r in reps], s=28)
ax[0].axvline(2.8, ls="--", c="k", lw=1)
ax[0].set_xlabel("gate-ligand min distance (Å)"); ax[0].set_ylabel("density"); ax[0].legend(fontsize=7)
ax[1].set_ylabel("per-replicate median (Å)"); ax[1].tick_params(axis="x", labelrotation=20)
fig.tight_layout(); fig.savefig("fig_holo_gate_distance.png", dpi=300)
print("\nSaved fig_holo_gate_distance.png")

## 7. What you may and may not conclude

**Supported:** the equilibrium distribution of gate-to-ligand distance in explicit solvent, and whether the
ligand stays bound. That is a direct dynamical test of the steric-gate hypothesis.

**Not supported:** a binding free energy. `ddG_bind` needs alchemical free-energy or MM/GBSA with a proper
entropy treatment, neither of which runs here. Do not reintroduce kcal/mol binding numbers into the
manuscript on the strength of these trajectories.

**Minimum reporting standard:** state the ligand protonation state, the OpenFF version, the charge method,
and the number of independent replicates. One run per system gives no error bar - if the gate difference is
small, prefer 3 x 3 ns replicates per system over 1 x 10 ns.